# DougScore source audit

## tl;dr

The supplied CSV has **585 vehicle reviews**, model years **1954–2023**, with complete year/make/model and ten category scores. All category values and all three totals pass validation; no normalized year/make/raw-model duplicate keys occur. Seventeen make names have surrounding whitespace. There are no separate trim, review publication date, or score-version date columns. The 584 populated “Video Link” values are time-formatted text, with no usable HTTP(S) URLs.

This snapshot supports a separate expert-score baseline. It does not yet support a verified cohort join or historical price prediction.


## Context & Methods

Question: how can this expert-review snapshot complement the project's YouTube sentiment scoring?

Input: the user-supplied DougScore CSV. The audit reads it without changing it and records its SHA-256. Checks cover layout, completeness, score ranges, total reconciliation, duplicate keys, text normalization, and identifier/date availability. The two header rows and one blank row are explicitly recognized.

### Key Assumptions

The first three columns represent vehicle model year, make, and the reviewed model/variant name. CSV logical row numbers locate records; they are not durable review IDs. Model names such as “4Runner TRD Pro” contain variant information that has not been mapped to vPIC.

DougScore category definitions: [Doug DeMuro's methodology](https://www.dougdemuro.com/dougscore), consulted 2026-10-01. The category-to-aspect mapping and integration decisions are in [DOUGSCORE_INTEGRATION_PLAN.md](DOUGSCORE_INTEGRATION_PLAN.md).

No vPIC or YouTube join is performed here, so overlap and join coverage remain unmeasured. Source model-year coverage does not establish review dates or freshness. The source snapshot is not copied into a model-development table.


## Data

Change `SOURCE` and `EXPECTED_SHA256` deliberately when auditing a different snapshot. Only Python's standard library is required. Saved cell outputs come from executing these cells sequentially with the bundled Python runtime.


In [1]:
import csv
import hashlib
import io
import json
import re
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

SOURCE = Path(r"C:/Users/danie/Downloads/DougScore - DougScore.csv")
EXPECTED_SHA256 = "5b93a406ad014c7b473b3dcef69360d69c93249bf89815416899b373030f1600"
source_bytes = SOURCE.read_bytes()
source_sha256 = hashlib.sha256(source_bytes).hexdigest()
assert source_sha256 == EXPECTED_SHA256, "Source changed: audit the new snapshot explicitly."
executed_at_utc = datetime.now(timezone.utc).isoformat()
print(json.dumps({"source": str(SOURCE), "bytes": len(source_bytes), "sha256": source_sha256, "executed_at_utc": executed_at_utc}, indent=2))


{
  "source": "C:\\Users\\danie\\Downloads\\DougScore - DougScore.csv",
  "bytes": 51365,
  "sha256": "5b93a406ad014c7b473b3dcef69360d69c93249bf89815416899b373030f1600",
  "executed_at_utc": "2026-10-01T22:45:27.101182+00:00"
}


In [2]:
columns = [
    "model_year_raw", "make_raw", "model_variant_raw",
    "styling", "acceleration", "handling", "fun_factor", "cool_factor", "weekend_total",
    "features", "comfort", "quality", "practicality", "value", "daily_total", "dougscore",
    "video_link_raw", "filming_city_raw", "filming_region_raw", "vehicle_country_raw",
]
raw_rows = list(csv.reader(io.StringIO(source_bytes.decode("utf-8-sig"))))
assert len(raw_rows) >= 4
assert raw_rows[0][0] == "Welcome to the DougScore!"
assert raw_rows[1][3:8] == ["Styling", "Acceleration", "Handling", "Fun Factor", "Cool Factor"]
assert raw_rows[1][9:14] == ["Features", "Comfort", "Quality", "Practicality", "Value"]
assert not any(value.strip() for value in raw_rows[2])
assert all(len(row) == len(columns) for row in raw_rows), "Unexpected row width."
records = []
for csv_row_number, row in enumerate(raw_rows[3:], start=4):
    assert re.fullmatch(r"\d{4}", row[0].strip()), f"Unexpected model year at row {csv_row_number}"
    records.append({"csv_row_number": csv_row_number, **dict(zip(columns, row))})
assert records, "No vehicle rows."
print(json.dumps({"raw_logical_rows": len(raw_rows), "header_rows": 2, "blank_rows": 1, "vehicle_rows": len(records), "columns": len(columns), "preview": [{k: r[k] for k in ["csv_row_number", "model_year_raw", "make_raw", "model_variant_raw", "dougscore"]} for r in records[:3]]}, indent=2))


{
  "raw_logical_rows": 588,
  "header_rows": 2,
  "blank_rows": 1,
  "vehicle_rows": 585,
  "columns": 20,
  "preview": [
    {
      "csv_row_number": 4,
      "model_year_raw": "2020",
      "make_raw": "McLaren",
      "model_variant_raw": "Speedtail",
      "dougscore": "74"
    },
    {
      "csv_row_number": 5,
      "model_year_raw": "1994",
      "make_raw": "McLaren",
      "model_variant_raw": "F1",
      "dougscore": "74"
    },
    {
      "csv_row_number": 6,
      "model_year_raw": "2022",
      "make_raw": "Lucid",
      "model_variant_raw": "Air Dream Edition",
      "dougscore": "73"
    }
  ]
}


## Results

### Score validity and duplicate keys

Each category must be an integer from 1 through 10. Weekend and daily totals must equal their five component scores, and DougScore must equal the two totals. Duplicate-key checks normalize whitespace and case, without guessing manufacturer aliases or splitting variants.


In [3]:
weekend_categories = ["styling", "acceleration", "handling", "fun_factor", "cool_factor"]
daily_categories = ["features", "comfort", "quality", "practicality", "value"]
score_categories = weekend_categories + daily_categories
invalid_score_rows = []
total_mismatch_rows = []
normalized_keys = defaultdict(list)
for record in records:
    row_number = record["csv_row_number"]
    normalized_keys[tuple(record[k].strip().casefold() for k in ["model_year_raw", "make_raw", "model_variant_raw"])].append(row_number)
    try:
        scores = {name: int(record[name]) for name in score_categories}
        totals = {name: int(record[name]) for name in ["weekend_total", "daily_total", "dougscore"]}
    except ValueError:
        invalid_score_rows.append(row_number)
        continue
    if not all(1 <= score <= 10 for score in scores.values()):
        invalid_score_rows.append(row_number)
    if (sum(scores[k] for k in weekend_categories) != totals["weekend_total"]
        or sum(scores[k] for k in daily_categories) != totals["daily_total"]
        or totals["weekend_total"] + totals["daily_total"] != totals["dougscore"]):
        total_mismatch_rows.append(row_number)
duplicate_key_rows = [row_numbers for row_numbers in normalized_keys.values() if len(row_numbers) > 1]
exact_duplicate_rows = len(records) - len({tuple(record[k] for k in columns) for record in records})
missing_by_column = {k: sum(not record[k].strip() for record in records) for k in columns}
score_ranges = {k: [min(int(record[k]) for record in records), max(int(record[k]) for record in records)] for k in score_categories + ["weekend_total", "daily_total", "dougscore"]}
print(json.dumps({"invalid_score_rows": invalid_score_rows, "total_mismatch_rows": total_mismatch_rows, "normalized_duplicate_keys": duplicate_key_rows, "exact_duplicate_row_count": exact_duplicate_rows, "missing_by_column": missing_by_column, "score_ranges": score_ranges}, indent=2))


{
  "invalid_score_rows": [],
  "total_mismatch_rows": [],
  "normalized_duplicate_keys": [],
  "exact_duplicate_row_count": 0,
  "missing_by_column": {
    "model_year_raw": 0,
    "make_raw": 0,
    "model_variant_raw": 0,
    "styling": 0,
    "acceleration": 0,
    "handling": 0,
    "fun_factor": 0,
    "cool_factor": 0,
    "weekend_total": 0,
    "features": 0,
    "comfort": 0,
    "quality": 0,
    "practicality": 0,
    "value": 0,
    "daily_total": 0,
    "dougscore": 0,
    "video_link_raw": 1,
    "filming_city_raw": 1,
    "filming_region_raw": 0,
    "vehicle_country_raw": 0
  },
  "score_ranges": {
    "styling": [
      2,
      10
    ],
    "acceleration": [
      1,
      10
    ],
    "handling": [
      1,
      10
    ],
    "fun_factor": [
      1,
      10
    ],
    "cool_factor": [
      1,
      10
    ],
    "features": [
      1,
      10
    ],
    "comfort": [
      1,
      10
    ],
    "quality": [
      1,
      10
    ],
    "practicality": [
     

### Identity, dates, and source links

The CSV has 20 columns, with no explicit trim, publication date, video ID, or score-version date. Its model field describes a reviewed variant, rather than a validated vPIC Model. The original Google Sheet may retain hyperlinks that this CSV does not preserve; that has not been checked.


In [4]:
years = [int(record["model_year_raw"]) for record in records]
raw_makes = Counter(record["make_raw"] for record in records)
trimmed_makes = Counter(record["make_raw"].strip().casefold() for record in records)
make_whitespace_rows = [record["csv_row_number"] for record in records if record["make_raw"] != record["make_raw"].strip()]
video_link_formats = Counter(
    "blank" if not record["video_link_raw"].strip()
    else "http_url" if re.match(r"^https?://", record["video_link_raw"].strip(), re.IGNORECASE)
    else "time_formatted_text" if re.fullmatch(r"\d+:\d{2}(?::\d{2})?", record["video_link_raw"].strip())
    else "other_text"
    for record in records
)
year_bands = Counter("before_2010" if year < 2010 else "2010_through_2020" if year <= 2020 else "after_2020" for year in years)
identity_examples = [{k: r[k] for k in ["csv_row_number", "model_year_raw", "make_raw", "model_variant_raw"]} for r in records if r["make_raw"] in ["Mecedes-AMG", "Mercedes - Benz", "Toyota "]][:6]
profile = {
    "source_sha256": source_sha256,
    "executed_at_utc": executed_at_utc,
    "vehicle_rows": len(records), "raw_rows": len(raw_rows), "source_columns": len(columns),
    "model_year_min": min(years), "model_year_max": max(years),
    "make_labels_raw": len(raw_makes), "make_labels_trimmed_casefold": len(trimmed_makes),
    "make_whitespace_row_count": len(make_whitespace_rows),
    "invalid_score_row_count": len(invalid_score_rows), "total_mismatch_row_count": len(total_mismatch_rows),
    "normalized_duplicate_key_count": len(duplicate_key_rows), "exact_duplicate_row_count": exact_duplicate_rows,
    "missing_by_column": missing_by_column, "video_link_formats": dict(video_link_formats),
    "model_year_bands": dict(year_bands), "score_ranges": score_ranges,
    "explicit_trim_available": False, "review_published_at_available": False,
    "score_version_at_available": False, "join_coverage_measured": False,
}
print(json.dumps(profile, indent=2))
print("Examples needing identity review:")
print(json.dumps(identity_examples, indent=2))
assert len(records) == len(normalized_keys), "Duplicate-key remediation is needed."
assert not invalid_score_rows and not total_mismatch_rows, "Score remediation is needed."
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest() == source_sha256, "Source changed during audit."


{
  "source_sha256": "5b93a406ad014c7b473b3dcef69360d69c93249bf89815416899b373030f1600",
  "executed_at_utc": "2026-10-01T22:45:27.101182+00:00",
  "vehicle_rows": 585,
  "raw_rows": 588,
  "source_columns": 20,
  "model_year_min": 1954,
  "model_year_max": 2023,
  "make_labels_raw": 93,
  "make_labels_trimmed_casefold": 84,
  "make_whitespace_row_count": 17,
  "invalid_score_row_count": 0,
  "total_mismatch_row_count": 0,
  "normalized_duplicate_key_count": 0,
  "exact_duplicate_row_count": 0,
  "missing_by_column": {
    "model_year_raw": 0,
    "make_raw": 0,
    "model_variant_raw": 0,
    "styling": 0,
    "acceleration": 0,
    "handling": 0,
    "fun_factor": 0,
    "cool_factor": 0,
    "weekend_total": 0,
    "features": 0,
    "comfort": 0,
    "quality": 0,
    "practicality": 0,
    "value": 0,
    "daily_total": 0,
    "dougscore": 0,
    "video_link_raw": 1,
    "filming_city_raw": 1,
    "filming_region_raw": 0,
    "vehicle_country_raw": 0
  },
  "video_link_formats": {

## Takeaways

Use this file as an expert-only category baseline, keeping Daily, Weekend, and overall DougScore visible. Build a reviewed identity crosswalk before joining to vehicle cohorts; retain raw make and model/variant names. Restore review video URLs/IDs and publication metadata before pairing with YouTube comments. Publication dates alone cannot establish when a mutable score first became available.

Compare expert and community views side by side on supported aspects. Keep unmeasured aspects and unmatched vehicles unavailable. Do not use numeric expert scores as comment-polarity labels, or treat brand-level sentiment as an opinion about the exact reviewed variant. Historical price-model features require verified outcome cutoffs and score availability/version evidence.

No scores or mappings have been added to model training by this audit. See the integration plan for the proposed tables and execution sequence.
